# Flutter üreteci — v8 (kanıtlı) QLoRA

v7 ev stilini modelin kendi Flutter hafızasından öğretti. v8 her satırın önüne
numaralı bir **kanıt bloğu** koyuyor, böylece model güncel API bilgisini
okumayı öğreniyor — spec'in "üretmeden önce araştırır" vaadinin eğitim tarafı.

Cevaplar v7'den değiştirilmedi; yalnızca kullanıcı mesajı yeniden yazıldı.
Yani v8 kodlama tavanını yükseltmiyor, modelin bilgisini **tazelenebilir**
yapıyor.

Sıra: temel model ölçümü (öncesi) → eğitim → adapter ölçümü (sonrası).
Sonucu belirleyen tek sayı `followed_unseen`: eğitimde hiç görmediği bir API
göçünü kanıta bakarak yapabiliyor mu.


In [ ]:
import subprocess, sys, torch, os, shutil, glob

assert torch.cuda.is_available(), "GPU acik degil - Settings > Accelerator > GPU T4"
cap = torch.cuda.get_device_capability(0)
print("GPU:", torch.cuda.get_device_name(0), "sm_%d%d" % cap)
print("bellek: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1024**3))

# Fail here, in five seconds, rather than after an 8 GB download. A P100 is
# sm_60: Kaggle's torch build does not support it at all, and bitsandbytes needs
# sm_75 for 4-bit NF4. The first run of this notebook landed on one because
# kernel-metadata omitted machine_shape, and the error arrived thirty minutes in
# wearing a different mask.
assert cap >= (7, 5), (
    f"sm_{cap[0]}{cap[1]} yetersiz - 4-bit NF4 icin T4 (sm_75) gerekiyor. "
    "Settings > Accelerator > GPU T4 x2")


In [ ]:
# Qwen3 icin transformers >= 4.51 gerekiyor; Kaggle imaji eskiyse sessizce
# 'unknown architecture' ile duser.
!pip -q install -U "transformers>=4.51" "peft>=0.11" "bitsandbytes>=0.43" "accelerate>=0.30" datasets 2>&1 | tail -2
import transformers, peft, bitsandbytes
print("transformers", transformers.__version__, "| peft", peft.__version__, "| bnb", bitsandbytes.__version__)


In [ ]:
# Found rather than hard-coded: the first run failed because the dataset was
# still processing its new version when the kernel started, and a missing mount
# point reads as a plain FileNotFoundError with nothing pointing at the cause.
for root, dirs, files in os.walk("/kaggle/input"):
    print(root, "->", sorted(files)[:4], "..." if len(files) > 4 else "")
    if root.count("/") > 5:
        dirs.clear()

# Recursive, because the mount layout is not a promise: this notebook has seen
# the dataset appear directly under /kaggle/input and, on the next run, one
# level deeper under /kaggle/input/datasets. Searching for a file we know the
# name of survives both.
hits = glob.glob("/kaggle/input/**/flutter_screens_train_v8.jsonl", recursive=True)
assert hits, ("v8 veri seti bagli degil. Kaggle > Notebook > Add Input > "
              "emrahik/flutter-dataset (surumun islenmesi bitmis olmali)")
SRC, WORK = os.path.dirname(hits[0]), "/kaggle/working"
print("kaynak:", SRC)

os.makedirs(f"{WORK}/data", exist_ok=True)
for f in os.listdir(SRC):
    dst = f"{WORK}/data/{f}" if f.endswith(".jsonl") else f"{WORK}/{f}"
    shutil.copy(f"{SRC}/{f}", dst)
os.chdir(WORK)
print(sorted(os.listdir(WORK)))
print(sorted(os.listdir(f"{WORK}/data")))


## 1. Öncesi — temel model

Adapter yok. Bu sayılar karşılaştırma tabanı; `followed_*` burada düşük
çıkmalı, çünkü temel model kanıt bloğunu takip etmeye dair hiçbir şey
görmedi.


In [ ]:
!python flutter_eval.py --backend hf \
    --base-model Qwen/Qwen3-4B-Instruct-2507 \
    --eval data/flutter_screens_eval_v8.jsonl \
    --meta data/flutter_screens_eval_v8_meta.jsonl \
    --max-new-tokens 1200 \
    --dump before.jsonl


## 2. Eğitim

115 satır, 6 epoch, effective batch 8 → ~86 optimizer adımı.
`max-seq-len 2560` ölçülerek seçildi: v8'in en uzun satırı ~2130 token,
2048 bir satırı kırpıyor, 1536 satırların %15'ini.


In [ ]:
!python train_qlora_qwen.py \
    --train data/flutter_screens_train_v8.jsonl \
    --eval  data/flutter_screens_eval_v8.jsonl \
    --out-dir out/flutter-v8 \
    --max-seq-len 2560 \
    --epochs 6 --grad-accum 8


## 3. Sonrası — adapter ile

Aynı promptlar, aynı greedy çözme. Tek fark model.


In [ ]:
!python flutter_eval.py --backend hf \
    --base-model Qwen/Qwen3-4B-Instruct-2507 \
    --adapter out/flutter-v8 \
    --eval data/flutter_screens_eval_v8.jsonl \
    --meta data/flutter_screens_eval_v8_meta.jsonl \
    --max-new-tokens 1200 \
    --dump after.jsonl


## 4. Çıktı

Adapter `out/flutter-v8/` altında, birkaç on MB. Kaggle output'undan
indirilir; merge ve MLC derlemesi ayrı bir adım.

`before.jsonl` / `after.jsonl` ham tamamlamaları taşıyor — sayı şüpheli
görünürse modelin ne yazdığına bakılacak yer orası.


In [ ]:
import json, glob
for f in ["out/flutter-v8/train_metrics.json"]:
    if os.path.exists(f):
        print(json.dumps(json.load(open(f)), indent=2)[:1200])
print()
!du -sh out/flutter-v8 2>/dev/null
!ls -la out/flutter-v8
